# Rubin DP2 light curves with LSDB

In this tutorial, you will learn:
- How to open DP2 catalogs and what a nested light-curve column is
- How to filter objects and their observations
- How to get light curves of known transients
- How to fit transients with a multi-band model
- How to extract light-curve features and select variable objects

In [ ]:
%pip install --quiet lsdb-rubin 'light-curve[full]'

In [ ]:
import warnings
from io import StringIO

import light_curve as licu
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dask.distributed import Client
from light_curve.light_curve_py.warnings import ExperimentalWarning
from lsdb_rubin.bands import band_names_ugrizy, band_wavelengths_ugrizy, plot_filter_colors_white_background
from lsdb_rubin.plot_light_curve import plot_light_curve
from upath import UPath

LSDB operations are lazy and run in parallel with Dask. We start a local Dask cluster with 4 workers, 1 thread each; every `compute()` below runs on it. Follow the progress in the dashboard.

In [ ]:
client = Client(n_workers=4, threads_per_worker=1)
client

## Targets

Ten transients from TNS detected in the alerts and in the First Look images.

In [ ]:
targets_csv = """ra,dec
187.4565,8.213469
186.016373037,8.4117596341
149.261451,1.291222
51.620551,-28.114117
10.581959,-45.278411
10.918621,-44.157346
10.178225,-45.842358
52.718802,-28.362480
61.964820,-48.713443
52.786108,-27.341361
"""
targets = lsdb.from_dataframe(pd.read_csv(StringIO(targets_csv)))
targets

## Rubin DIA objects

`dia_object_collection` has one row per DIA object. Its light curves are stored in nested columns:
- `diaSource`: detections on difference images
- `diaObjectForcedSource`: forced photometry at the DIA object position on all visits

Opening a catalog reads only its metadata. No data is loaded yet. We select only the columns we need, including nested ones.

In [ ]:
dia = lsdb.open_catalog(
    UPath("/rubin/lsdb_data") / "dia_object_collection",
    columns=[
        "diaObjectId",
        "diaObjectForcedSource.band",
        "diaObjectForcedSource.midpointMjdTai",
        "diaObjectForcedSource.psfDiffFlux",
        "diaObjectForcedSource.psfDiffFluxErr_corrected",
        "diaObjectForcedSource.psfDiffFlux_flag",
        "diaObjectForcedSource.diff_PixelFlags_nodataCenter",
        "diaObjectForcedSource.pixelFlags_saturatedCenter",
        "diaObjectForcedSource.invalidPsfFlag",
    ],
)
dia

## Quality cuts

`query` on nested columns filters observations inside each light curve. The objects themselves are not removed: a light curve with no observations left becomes empty (`None`).

In [ ]:
dia = dia.query(
    "~diaObjectForcedSource.psfDiffFlux_flag"
    " & ~diaObjectForcedSource.diff_PixelFlags_nodataCenter"
    " & ~diaObjectForcedSource.pixelFlags_saturatedCenter"
    " & ~diaObjectForcedSource.invalidPsfFlag"
)

`map_partitions` applies any function to each partition, a `NestedFrame`. Here we use it to drop objects with empty light curves.

In [ ]:
dia = dia.map_partitions(lambda df: df.dropna(subset="diaObjectForcedSource"))

## Crossmatch

The target list is small, so it goes on the left side of the crossmatch. The DIA collection provides the margin catalog.

In [ ]:
matched = targets.crossmatch(dia, radius_arcsec=1, suffixes=("_tns", "_dia"), suffix_method="overlapping_columns")
matched

In [ ]:
df = matched.compute()
df

Each row is one DIA object, and its light curve is a data frame nested in that row:

In [ ]:
df["diaObjectForcedSource"].iloc[0]

## Multi-band model fit

We use the [`light-curve`](https://light-curve.snad.space) package. Its [`RainbowFit`](https://light-curve.snad.space/latest/features/rainbow/) fits all bands at once with a blackbody spectrum of evolving temperature and a rising and fading bolometric light curve ([Russeil et al. 2024](https://doi.org/10.1051/0004-6361/202348158)).

`map_rows` applies a function to each object, nested columns come in as numpy arrays.

In [ ]:
warnings.filterwarnings("ignore", category=ExperimentalWarning)
rainbow = licu.RainbowFit.from_nm(band_wavelengths_ugrizy, with_baseline=True)


def fit_rainbow(band, t, flux, err):
    # light-curve needs time-sorted arrays of the same dtype
    order = np.argsort(t)
    t, flux, err = (np.asarray(x[order], dtype=np.float64) for x in (t, flux, err))
    try:
        with np.errstate(all="ignore"):
            params = rainbow(t, flux, sigma=err, band=band[order])
    except RuntimeError:  # fit failed
        params = np.full(len(rainbow.names), np.nan)
    return dict(zip(rainbow.names, params))

In [ ]:
LC_COLUMNS = [
    "diaObjectForcedSource.band",
    "diaObjectForcedSource.midpointMjdTai",
    "diaObjectForcedSource.psfDiffFlux",
    "diaObjectForcedSource.psfDiffFluxErr_corrected",
]

df = df.map_rows(fit_rainbow, columns=LC_COLUMNS, row_container="args", append_columns=True)
df[["diaObjectId", "reference_time", "rise_time", "fall_time", "T"]]

## Light curves

`lsdb-rubin` plots Rubin light curves; we overlay the model on top.

In [ ]:
def plot_dia_light_curves(df, rainbow=None):
    for _, row in df.iterrows():
        lc = row["diaObjectForcedSource"]
        plt.figure(figsize=(10, 4))
        plot_light_curve(lc, flux_field="psfDiffFlux", corrected_err=True, title=f"diaObjectId {row['diaObjectId']}")
        if rainbow is not None:
            t = np.linspace(lc["midpointMjdTai"].min(), lc["midpointMjdTai"].max(), 1000)
            for band in lc["band"].unique():
                flux = rainbow.model(t, np.full(t.size, band), *row[rainbow.names])
                plt.plot(t, flux, color=plot_filter_colors_white_background[band])
        plt.show()


plot_dia_light_curves(df.dropna(subset="reference_time"), rainbow)

## Light-curve features

Now let's find variable objects ourselves, in a field around one of the targets.

`light-curve` features become [multi-band](https://light-curve.snad.space/latest/features/multiband/) when given `bands`: each band is evaluated separately, giving `chi2_u`, `chi2_g`, etc.

In [ ]:
extractor = licu.Extractor(
    licu.ReducedChi2(bands=band_names_ugrizy),
    licu.ObservationCount(bands=band_names_ugrizy),
)


def extract_features(band, t, flux, err):
    order = np.argsort(t)
    t, flux, err = (np.asarray(x[order], dtype=np.float64) for x in (t, flux, err))
    # fill_value is used for bands with too few observations
    values = extractor(t, flux, err, band=band[order], fill_value=np.nan)
    return dict(zip(extractor.names, values))

`map_rows` works lazily on LSDB catalogs too; we need to provide `meta` describing the output columns.

In [ ]:
field = dia.cone_search(ra=52.718802, dec=-28.362480, radius_arcsec=600)
features = field.map_rows(
    extract_features,
    columns=LC_COLUMNS,
    row_container="args",
    meta={name: np.float64 for name in extractor.names},
    append_columns=True,
)
features

Features are ordinary columns, so we select on them with `query`: objects variable in all of *g*, *r* and *i*.

In [ ]:
variables = features.query("chi2_g > 10 and chi2_r > 10 and chi2_i > 10")
var_df = variables.compute().nlargest(5, "chi2_r")
var_df[["diaObjectId", "chi2_g", "chi2_r", "chi2_i", "observation_count_r"]]

In [ ]:
plot_dia_light_curves(var_df)

In [ ]:
client.close()